In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, regexp_extract

def get_available_years(path: str) -> list[str]:
    """
    Retorna todos os os anos disponíveis para uma tabela.

    Exemplo:
        get_available_years(f"{BRONZE_PATH}/TS_ALUNO")

    Retorno:
        ['2023', '2024', '2025']
    """

    return sorted(
        [
            folder.name.rstrip("/")
            for folder in dbutils.fs.ls(path)
            if folder.isDir()
        ]
    )

def read(
    base_path: str,
    table_name: str,
    format: str = "csv",
    recursive_by_year: bool = False
) -> DataFrame:
    """
    Lê uma tabela do Data Lake.

    Parameters
    ----------
    base_path : str
        Caminho base da camada (BRONZE_PATH, SILVER_PATH, GOLD_PATH...)

    table_name : str
        Nome da tabela.

    format : str
        csv, delta ou parquet.

    recursive_by_year : bool
        Quando True, lê todos os anos disponíveis e realiza
        unionByName entre eles.

    Returns
    -------
    pyspark.sql.DataFrame
    """

    format = format.lower()

    if recursive_by_year:

        years = get_available_years(f"{base_path}/{table_name}")

        dfs = []

        for year in years:

            path = f"{base_path}/{table_name}/{year}"

            if format == "csv":

                df = (
                    spark.read
                    .options(**CSV_OPTIONS)
                    .csv(path)
                )

            else:

                df = (
                    spark.read
                    .format(format)
                    .load(path)
                )

            df = df.withColumn("ANO_REFERENCIA", lit(int(year)))

            dfs.append(df)

        if not dfs:
            raise Exception(f"Nenhum arquivo encontrado em {base_path}/{table_name}")

        return reduce(
            lambda df1, df2: df1.unionByName(
                df2,
                allowMissingColumns=True
            ),
            dfs
        )

    else:

        path = f"{base_path}/{table_name}"

        if format == "csv":

            return (
                spark.read
                .options(**CSV_OPTIONS)
                .csv(path)
            )

        return (
            spark.read
            .format(format)
            .load(path)
        )